# Yoga Dataset Generation

This notebook generates a comprehensive yoga dataset using LLM API.

**Requirements:**

- Generate 108+ yoga poses
- Cover all categories: standing, seated, balancing, inversion, backbend, forward_fold, twist, restorative
- Include all difficulty levels: beginner, intermediate, advanced
- Populate all required fields for each pose


In [3]:
import os
import json
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
from typing import List, Dict

In [ ]:
# Load environment variables
load_dotenv()

# API Configuration
api_key = os.getenv("HYPERBOLIC_API_KEY")
model = os.getenv("LLM_MODEL", "meta-llama/Meta-Llama-3.1-70B-Instruct")

# Initialize OpenAI client with Hyperbolic endpoint
client = OpenAI(api_key=api_key, base_url="https://api.hyperbolic.xyz/v1")

print(f"Using model: {model}")
print(f"API configured: {api_key is not None}")

Using model: meta-llama/Meta-Llama-3.1-70B-Instruct
API configured: True


## Yoga Pose Schema

Each yoga pose record contains the following fields:

- **id**: Unique identifier (int)
- **pose_name**: English name (str)
- **sanskrit_name**: Traditional Sanskrit name (str)
- **category**: Type of pose (str) - standing/seated/balancing/inversion/backbend/forward_fold/twist/restorative
- **difficulty_level**: Skill level (str) - beginner/intermediate/advanced
- **benefits**: Physical and mental benefits (str)
- **contraindications**: When to avoid this pose (str)
- **breathing_pattern**: Inhale/exhale cues (str)
- **duration_or_reps**: How long to hold or how many repetitions (str)
- **modifications**: Easier variations or props (str)
- **instructions**: Step-by-step guide (str)


In [ ]:
# Define yoga pose categories and difficulty levels
CATEGORIES = [
    "standing",
    "seated",
    "balancing",
    "inversion",
    "backbend",
    "forward_fold",
    "twist",
    "restorative",
]

DIFFICULTY_LEVELS = ["beginner", "intermediate", "advanced"]

# Target distribution: 108 poses total
# ~13-14 poses per category
# Mix of difficulty levels in each category
TARGET_POSES = 108

## Generate 108 Distinct Pose Names First

Generate all pose names upfront to ensure uniqueness, then generate details.


In [25]:
def generate_108_pose_names():
    """
    Generate 108 distinct yoga pose names with their categories and difficulty levels.
    """
    prompt = """Generate a list of exactly 108 distinct yoga poses.

For each pose, provide:
- pose_name: English name (must be unique)
- sanskrit_name: Sanskrit name
- category: one of [standing, seated, balancing, inversion, backbend, forward_fold, twist, restorative]
- difficulty_level: one of [beginner, intermediate, advanced]

Distribution:
- ~13-14 poses per category
- 40% beginner, 40% intermediate, 20% advanced

Return ONLY a JSON array. No duplicates allowed.

Example:
[{
  "pose_name": "Mountain Pose",
  "sanskrit_name": "Tadasana",
  "category": "standing",
  "difficulty_level": "beginner"
}]
"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "You are a yoga expert. Generate exactly 108 unique yoga poses.",
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0.8,
        max_tokens=12000,  # Increased from 6000 to allow full generation
    )

    content = response.choices[0].message.content.strip()

    # Extract JSON
    if content.startswith("```"):
        content = content.split("```")[1]
        if content.startswith("json"):
            content = content[4:]
        content = content.strip()

    poses = json.loads(content)

    # Check if we got 108 poses
    if len(poses) < 108:
        print(f"⚠️ Warning: Only generated {len(poses)} poses instead of 108")
        print(f"This likely means the model hit the token limit.")
        print(
            f"You may need to run this function multiple times or increase max_tokens further."
        )

    return poses

In [ ]:
# Generate pose names
pose_names = generate_108_pose_names()
print(f"Generated {len(pose_names)} pose names")

Generated 82 pose names


In [ ]:
# Check for duplicates
names = [p["pose_name"] for p in pose_names]
duplicates = len(names) - len(set(names))
print(f"Duplicates: {duplicates}")

Duplicates: 0


In [ ]:
if duplicates > 0:
    print("\n⚠️ Removing duplicates...")
    seen = set()
    unique_poses = []
    for pose in pose_names:
        if pose["pose_name"] not in seen:
            seen.add(pose["pose_name"])
            unique_poses.append(pose)
    pose_names = unique_poses
    print(f"After dedup: {len(pose_names)} poses")

In [28]:
# Show sample
print("\nSample poses:")
for i in range(min(10, len(pose_names))):
    p = pose_names[i]
    print(
        f"  {i+1}. {p['pose_name']} ({p['sanskrit_name']}) - {p['category']}/{p['difficulty_level']}"
    )


Sample poses:
  1. Mountain Pose (Tadasana) - standing/beginner
  2. Downward-Facing Dog (Adho Mukha Svanasana) - standing/beginner
  3. Warrior Pose (Virabhadrasana) - standing/beginner
  4. Triangle Pose (Trikonasana) - standing/beginner
  5. Tree Pose (Vrksasana) - standing/beginner
  6. Eagle Pose (Garudasana) - standing/beginner
  7. Seated Forward Fold (Paschimottanasana) - seated/beginner
  8. Seated Twist (Bharadvajasana) - seated/beginner
  9. Seated Spinal Twist (Bharadvajasana II) - seated/beginner
  10. Seated Hero Pose (Virasana) - seated/beginner


In [ ]:
def generate_batch(count, existing_names, category=None, difficulty=None):
    """Generate a small batch of poses, avoiding existing names."""

    # Create exclusion list
    exclusion = ", ".join(existing_names[:30]) if existing_names else "None"

    category_filter = (
        f"Category: {category}"
        if category
        else "Any category from: standing, seated, balancing, inversion, backbend, forward_fold, twist, restorative"
    )
    difficulty_filter = (
        f"Difficulty: {difficulty}"
        if difficulty
        else "Any difficulty: beginner, intermediate, or advanced"
    )

    prompt = f"""Generate exactly {count} unique yoga poses.

{category_filter}
{difficulty_filter}

AVOID these existing poses: {exclusion}

For each pose provide:
- pose_name: English name (MUST be unique)
- sanskrit_name: Sanskrit name
- category: one of [standing, seated, balancing, inversion, backbend, forward_fold, twist, restorative]
- difficulty_level: one of [beginner, intermediate, advanced]

Return ONLY a JSON array of {count} poses.

Example:
[{{"pose_name": "Extended Side Angle", "sanskrit_name": "Utthita Parsvakonasana", "category": "standing", "difficulty_level": "intermediate"}}]
"""

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": "You are a yoga expert. Generate unique yoga poses.",
                },
                {"role": "user", "content": prompt},
            ],
            temperature=0.9,
            max_tokens=2000,
        )

        content = response.choices[0].message.content.strip()

        # Extract JSON
        if content.startswith("```"):
            content = content.split("```")[1]
            if content.startswith("json"):
                content = content[4:]
            content = content.strip()

        poses = json.loads(content)
        return poses

    except Exception as e:
        print(f"Error: {e}")
        return []

In [ ]:
108 - len(pose_names)

26

In [34]:
additional_pose_names = generate_batch(30, names)
print(f"Generated {len(additional_pose_names)} additional pose names")

Generated 33 additional pose names


In [35]:
additional_pose_names[1]

{'pose_name': 'Sky Dancer',
 'sanskrit_name': 'Akasha Nritya',
 'category': 'balancing',
 'difficulty_level': 'advanced'}

In [43]:
total_poses = pose_names + additional_pose_names
print(f"Total poses: {len(total_poses)}")

Total poses: 115


In [45]:
# Check for duplicates
names = [p["pose_name"] for p in total_poses]
duplicates = len(names) - len(set(names))
print(f"Duplicates: {duplicates}")

if duplicates > 0:
    print("\n⚠️ Removing duplicates...")
    seen = set()
    unique_poses = []
    for pose in pose_names:
        if pose["pose_name"] not in seen:
            seen.add(pose["pose_name"])
            unique_poses.append(pose)
    pose_names = unique_poses
    print(f"After dedup: {len(pose_names)} poses")

Duplicates: 0


In [46]:
# Show sample
print("\nSample poses:")
for i in range(min(10, len(total_poses))):
    p = pose_names[i]
    print(
        f"  {i+1}. {p['pose_name']} ({p['sanskrit_name']}) - {p['category']}/{p['difficulty_level']}"
    )


Sample poses:
  1. Mountain Pose (Tadasana) - standing/beginner
  2. Downward-Facing Dog (Adho Mukha Svanasana) - standing/beginner
  3. Warrior Pose (Virabhadrasana) - standing/beginner
  4. Triangle Pose (Trikonasana) - standing/beginner
  5. Tree Pose (Vrksasana) - standing/beginner
  6. Eagle Pose (Garudasana) - standing/beginner
  7. Seated Forward Fold (Paschimottanasana) - seated/beginner
  8. Seated Twist (Bharadvajasana) - seated/beginner
  9. Seated Spinal Twist (Bharadvajasana II) - seated/beginner
  10. Seated Hero Pose (Virasana) - seated/beginner


In [47]:
def generate_pose_details(pose_name, sanskrit_name, category, difficulty):
    """
    Generate full details for a specific pose.
    """
    prompt = f"""Generate detailed information for this yoga pose:

Pose: {pose_name} ({sanskrit_name})
Category: {category}
Difficulty: {difficulty}

Provide:
- benefits: Physical and mental benefits (2-3 sentences)
- contraindications: When to avoid (1-2 sentences)
- breathing_pattern: Inhale/exhale cues
- duration_or_reps: How long to hold
- modifications: Easier variations or props
- instructions: Step-by-step guide (3-5 steps)

Return ONLY a JSON object:
{{
  "benefits": "...",
  "contraindications": "...",
  "breathing_pattern": "...",
  "duration_or_reps": "...",
  "modifications": "...",
  "instructions": "..."
}}
"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "You are a yoga expert providing detailed pose information.",
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0.7,
        max_tokens=1000,
    )

    content = response.choices[0].message.content.strip()

    # Extract JSON
    if content.startswith("```"):
        content = content.split("```")[1]
        if content.startswith("json"):
            content = content[4:]
        content = content.strip()

    details = json.loads(content)
    return details


print("Function defined: generate_pose_details()")

Function defined: generate_pose_details()


In [48]:
# Generate full details for all poses
import time

all_poses = []

for i, pose_info in enumerate(total_poses, 1):
    print(
        f"\rGenerating details for pose {i}/{len(total_poses)}: {pose_info['pose_name']}",
        end="",
    )

    try:
        details = generate_pose_details(
            pose_info["pose_name"],
            pose_info["sanskrit_name"],
            pose_info["category"],
            pose_info["difficulty_level"],
        )

        # Combine name info with details
        full_pose = {
            "id": i,
            "pose_name": pose_info["pose_name"],
            "sanskrit_name": pose_info["sanskrit_name"],
            "category": pose_info["category"],
            "difficulty_level": pose_info["difficulty_level"],
            **details,
        }

        all_poses.append(full_pose)

        # Small delay to avoid rate limiting
        time.sleep(0.5)

    except Exception as e:
        print(f"\nError on {pose_info['pose_name']}: {e}")
        continue

print(f"\n\nGenerated full details for {len(all_poses)} poses")

Generating details for pose 115/115: Echo ChamberiphsFold with a Twistt Twist

Generated full details for 115 poses


In [49]:
# Create DataFrame and save
df_yoga = pd.DataFrame(all_poses)

In [50]:
# Reorder columns
column_order = [
    "id",
    "pose_name",
    "sanskrit_name",
    "category",
    "difficulty_level",
    "benefits",
    "contraindications",
    "breathing_pattern",
    "duration_or_reps",
    "modifications",
    "instructions",
]
df_yoga = df_yoga[column_order]

In [51]:
print(f"Dataset shape: {df_yoga.shape}")
print(f"\nDuplicates: {df_yoga['pose_name'].duplicated().sum()}")
print(f"Missing values: {df_yoga.isnull().sum().sum()}")

print("\nCategory distribution:")
print(df_yoga["category"].value_counts().sort_index())

print("\nDifficulty distribution:")
print(df_yoga["difficulty_level"].value_counts())

Dataset shape: (115, 11)

Duplicates: 0
Missing values: 0

Category distribution:
category
backbend        11
balancing       17
forward_fold    12
inversion       10
restorative     19
seated          18
standing        20
twist            8
Name: count, dtype: int64

Difficulty distribution:
difficulty_level
intermediate    53
beginner        39
advanced        23
Name: count, dtype: int64


In [52]:
df_yoga

,id,pose_name,sanskrit_name,category,difficulty_level,benefits,contraindications,breathing_pattern,duration_or_reps,modifications,instructions
0,1,Mountain Pose,Tadasana,standing,beginner,"Mountain Pose (Tadasana) improves posture, bal...",Avoid Mountain Pose if you have severe ankle o...,Inhale: Lengthen the spine and feel the chest ...,"Hold the pose for 3-5 breaths, or use it as a ...","For those with ankle or knee issues, practice ...",1. Stand with your feet hip-width apart and pa...
1,2,Downward-Facing Dog,Adho Mukha Svanasana,standing,beginner,Downward-Facing Dog (Adho Mukha Svanasana) str...,Avoid this pose if you have high blood pressur...,"Inhale as you bend forward, lengthening the sp...","Hold the pose for 3-5 breaths, or 30-60 second...",For beginners or those with flexibility limita...,1. Start on all fours (tabletop position) with...
2,3,Warrior Pose,Virabhadrasana,standing,beginner,"Warrior Pose strengthens the legs, hips, and c...",Avoid Warrior Pose if you have any injuries or...,Inhale deeply as you step your right foot forw...,Hold Warrior Pose for 3-5 breaths on each side...,"For a modified version, you can shorten the st...","Step 1: Stand with your feet wide apart, with ..."
3,4,Triangle Pose,Trikonasana,standing,beginner,Triangle Pose (Trikonasana) offers a range of ...,Avoid this pose if you have a severe neck or h...,Inhale: Lengthen the spine and reach the arms ...,Hold the pose for 30 seconds to 1 minute on ea...,For beginners or those with flexibility limita...,"1. Stand with your feet wide apart, with one f..."
4,5,Tree Pose,Vrksasana,standing,beginner,"Tree Pose (Vrksasana) helps improve balance, s...",Avoid practicing Tree Pose if you have severe ...,Inhale: Lengthen the spine and engage the core...,"Hold the pose for 30 seconds to 1 minute, brea...","For beginners, you can start by practicing Tre...","1. Stand on one leg, with the other foot resti..."
...,...,...,...,...,...,...,...,...,...,...,...
110,111,Starlight Slip,Tara Sarana,forward_fold,intermediate,The Starlight Slip (Tara Sarana) forward fold ...,Avoid this pose if you have any severe back or...,Inhale: Lengthen the spine and reach the arms ...,"Hold the pose for 3-5 breaths, or 30-60 second...","For a more accessible version, place a block o...",1. Begin by standing with the feet hip-width a...
111,112,River Rock,Nadi Shila,balancing,advanced,River Rock (Nadi Shila) improves balance and f...,Avoid River Rock if you have severe ankle inju...,"Inhale as you balance and lengthen, exhale as ...","Hold River Rock for 3-5 breaths, repeat on the...","For beginners, practice near a wall for suppor...","1. Start in Mountain Pose, then shift your wei..."
112,113,Moon's Aura,Chandra Aavara,restorative,beginner,Moon's Aura (Chandra Aavara) provides physical...,Avoid this pose if you have any recent injurie...,"Inhale deeply, feeling the chest expand and th...","Hold the pose for 5-10 minutes, allowing the b...","To make the pose more accessible, use a block ...",1. Start by lying on your back with your knees...
113,114,New Horizon,Nava Akasha,standing,intermediate,New Horizon (Nava Akasha) is a balancing stand...,Avoid practicing New Horizon if you have sever...,"Inhale as you prepare to balance, exhale as yo...","Hold the pose for 3-5 breaths on each leg, rep...","If you're new to balancing poses, practice wit...",1. Stand with your feet hip-width apart and en...


## Save Dataset


In [55]:
# Ensure data directory exists
os.makedirs("../data", exist_ok=True)

In [53]:
# Save
df_yoga.to_csv("../data/yoga_data_115.csv", index=False)
print("\n✅ Saved clean dataset to ../data/yoga_data_115.csv")


✅ Saved clean dataset to ../data/yoga_data_115.csv


## Manual Review and Corrections


In [56]:
df = pd.read_csv("../data/yoga_data_115.csv")

print("=" * 60)
print("YOGA DATASET VALIDATION REPORT")
print("=" * 60)

print(f"\n✓ Total poses: {len(df)}")
print(f"✓ All required columns present: {len(df.columns) == 11}")
print(f"✓ No missing values: {df.isnull().sum().sum() == 0}")
print(f"✓ No duplicate poses: {df.duplicated().sum() == 0}")

print("\n" + "=" * 60)
print("DISTRIBUTION")
print("=" * 60)
print("\nCategories:")
print(df["category"].value_counts().to_string())
print("\nDifficulty levels:")
print(df["difficulty_level"].value_counts().to_string())

print("\n" + "=" * 60)
print("SAMPLE POSES")
print("=" * 60)

for i in range(min(3, len(df))):
    print(f"\n--- Pose {i+1} ---")
    print(f"Name: {df.iloc[i]['pose_name']}")
    print(f"Sanskrit: {df.iloc[i]['sanskrit_name']}")
    print(f"Category: {df.iloc[i]['category']}")
    print(f"Difficulty: {df.iloc[i]['difficulty_level']}")
    print(f"Benefits: {df.iloc[i]['benefits'][:150]}...")
    print(f"Instructions: {df.iloc[i]['instructions'][:150]}...")

print("\n" + "=" * 60)
print("QUALITY CHECKS")
print("=" * 60)

# Check for very short content
short_benefits = df[df["benefits"].str.len() < 50]
short_instructions = df[df["instructions"].str.len() < 50]

print(f"✓ Poses with short benefits (<50 chars): {len(short_benefits)}")
print(
    f"✓ Poses with short instructions (<50 chars): {len(short_instructions)}"
)

# Check Sanskrit names
no_sanskrit = df[df["sanskrit_name"].str.len() < 3]
print(f"✓ Poses missing Sanskrit names: {len(no_sanskrit)}")

print("\n" + "=" * 60)
print("VALIDATION COMPLETE")
print("=" * 60)
print(f"Location: ../data/yoga_data.csv")

YOGA DATASET VALIDATION REPORT

✓ Total poses: 115
✓ All required columns present: True
✓ No missing values: True
✓ No duplicate poses: True

DISTRIBUTION

Categories:
category
standing        20
restorative     19
seated          18
balancing       17
forward_fold    12
backbend        11
inversion       10
twist            8

Difficulty levels:
difficulty_level
intermediate    53
beginner        39
advanced        23

SAMPLE POSES

--- Pose 1 ---
Name: Mountain Pose
Sanskrit: Tadasana
Category: standing
Difficulty: beginner
Benefits: Mountain Pose (Tadasana) improves posture, balance, and core strength while promoting a sense of grounding and calmness. It also establishes good alig...
Instructions: 1. Stand with your feet hip-width apart and parallel to each other, with your weight evenly distributed on both feet. 2. Lengthen your spine, feeling ...

--- Pose 2 ---
Name: Downward-Facing Dog
Sanskrit: Adho Mukha Svanasana
Category: standing
Difficulty: beginner
Benefits: Downward-Faci

## Merge with old dataset


In [59]:
# Load both datasets
df1 = pd.read_csv("../data/yoga_data.csv")
df2 = pd.read_csv("../data/yoga_data_115.csv")

In [60]:
print("=" * 70)
print("DATASET COMPARISON")
print("=" * 70)

print(f"\nDataset 1 (yoga_data.csv): {len(df1)} poses")
print(f"Dataset 2 (yoga_data_115.csv): {len(df2)} poses")

DATASET COMPARISON

Dataset 1 (yoga_data.csv): 108 poses
Dataset 2 (yoga_data_115.csv): 115 poses


In [61]:
# Find overlapping poses (by pose name)
overlap = set(df1["pose_name"].str.lower()) & set(df2["pose_name"].str.lower())
print(f"\nOverlapping poses: {len(overlap)}")


Overlapping poses: 21


In [62]:
# Find unique poses in each dataset
unique_df1 = df1[
    ~df1["pose_name"].str.lower().isin(df2["pose_name"].str.lower())
]
unique_df2 = df2[
    ~df2["pose_name"].str.lower().isin(df1["pose_name"].str.lower())
]

print(f"Unique to dataset 1: {len(unique_df1)}")
print(f"Unique to dataset 2: {len(unique_df2)}")

Unique to dataset 1: 87
Unique to dataset 2: 94


In [63]:
# Show some examples of overlapping poses
if len(overlap) > 0:
    print(f"\nSample overlapping poses:")
    for name in list(overlap)[:10]:
        print(f"  - {name.title()}")


Sample overlapping poses:
  - Lord Of The Dance Pose
  - Tree Pose
  - Triangle Pose
  - Child'S Pose
  - Savasana
  - Seated Spinal Twist
  - Crow Pose
  - Mountain Pose
  - Seated Leg Stretch
  - Cat-Cow Pose


In [64]:
# Show unique poses from each dataset
if len(unique_df1) > 0:
    print(f"\nSample unique poses from dataset 1:")
    for name in unique_df1["pose_name"].head(10):
        print(f"  - {name}")

if len(unique_df2) > 0:
    print(f"\nSample unique poses from dataset 2:")
    for name in unique_df2["pose_name"].head(10):
        print(f"  - {name}")


Sample unique poses from dataset 1:
  - Warrior Pose I
  - Plank Pose
  - Seated Forward Fold With Legs Wide
  - Pigeon Pose
  - Side Angle Pose
  - Warrior Pose II
  - Seated Head to Knee Pose
  - Side Crow Pose
  - Seated Spinal Twist II
  - Four-Limbed Staff Pose

Sample unique poses from dataset 2:
  - Warrior Pose
  - Seated Forward Fold
  - Seated Twist
  - Seated Hero Pose
  - Seated Forward Bend
  - Seated Head to Knee
  - Seated Forward Fold with Arms Up
  - Eagle Pose on One Leg
  - One-Legged Tree Pose
  - Warrior II on One Leg


In [65]:
print("\n" + "=" * 70)
print("MERGING STRATEGY")
print("=" * 70)

# Merge strategy: Keep all from df1, add unique from df2
print(f"\nKeeping all {len(df1)} poses from dataset 1")
print(f"Adding {len(unique_df2)} unique poses from dataset 2")

# Combine datasets
merged = pd.concat([df1, unique_df2], ignore_index=True)

# Reassign IDs sequentially
merged["id"] = range(1, len(merged) + 1)

# Reorder columns to match expected schema
column_order = [
    "id",
    "pose_name",
    "sanskrit_name",
    "category",
    "difficulty_level",
    "benefits",
    "contraindications",
    "breathing_pattern",
    "duration_or_reps",
    "modifications",
    "instructions",
]
merged = merged[column_order]

print(f"\nMerged dataset: {len(merged)} poses")


MERGING STRATEGY

Keeping all 108 poses from dataset 1
Adding 94 unique poses from dataset 2

Merged dataset: 202 poses


In [66]:
# Validate merged dataset
print("\n" + "=" * 70)
print("VALIDATION")
print("=" * 70)

print(f"\nDuplicate pose names: {merged['pose_name'].duplicated().sum()}")
print(f"Missing values: {merged.isnull().sum().sum()}")

print(f"\nCategory distribution:")
print(merged["category"].value_counts().sort_index())

print(f"\nDifficulty distribution:")
print(merged["difficulty_level"].value_counts())


VALIDATION

Duplicate pose names: 0
Missing values: 0

Category distribution:
category
backbend        20
balancing       33
forward_fold    26
inversion       20
lunge            3
restorative     20
seated          17
standing        38
twist           25
Name: count, dtype: int64

Difficulty distribution:
difficulty_level
intermediate    75
beginner        69
advanced        58
Name: count, dtype: int64


In [68]:
# Save merged dataset
output_file = "../data/yoga_data_merged.csv"
merged.to_csv(output_file, index=False)

In [69]:
print("\n" + "=" * 70)
print("RESULT")
print("=" * 70)
print(f"\n✅ Merged dataset saved to: {output_file}")
print(f"Total poses: {len(merged)}")
print(f"  - From dataset 1: {len(df1)}")
print(f"  - From dataset 2: {len(unique_df2)}")
print(f"  - Overlapping (kept from dataset 1): {len(overlap)}")
print("=" * 70)


RESULT

✅ Merged dataset saved to: ../data/yoga_data_merged.csv
Total poses: 202
  - From dataset 1: 108
  - From dataset 2: 94
  - Overlapping (kept from dataset 1): 21


## Comprehensive Data Quality Checks

Let's perform thorough validation to ensure dataset quality.


In [70]:
# Reload the final dataset
df_final = pd.read_csv("../data/yoga_data_merged.csv")

print("=" * 70)
print("COMPREHENSIVE DATA QUALITY REPORT")
print("=" * 70)

# Basic stats
print(f"\n📊 BASIC STATISTICS")
print(f"Total poses: {len(df_final)}")
print(f"Total columns: {len(df_final.columns)}")
print(f"Memory usage: {df_final.memory_usage(deep=True).sum() / 1024:.2f} KB")

COMPREHENSIVE DATA QUALITY REPORT

📊 BASIC STATISTICS
Total poses: 202
Total columns: 11
Memory usage: 398.27 KB


In [71]:
# Check for duplicates
print(f"\n🔍 DUPLICATE CHECKS")
print(f"Duplicate rows (all columns): {df_final.duplicated().sum()}")
print(f"Duplicate pose names: {df_final['pose_name'].duplicated().sum()}")
print(
    f"Duplicate Sanskrit names: {df_final['sanskrit_name'].duplicated().sum()}"
)
print(f"Duplicate IDs: {df_final['id'].duplicated().sum()}")

# Show any duplicate pose names
if df_final["pose_name"].duplicated().sum() > 0:
    print("\n⚠️ Duplicate pose names found:")
    duplicates = df_final[df_final["pose_name"].duplicated(keep=False)]
    print(
        duplicates[
            ["id", "pose_name", "category", "difficulty_level"]
        ].sort_values("pose_name")
    )
else:
    print("✓ No duplicate pose names")


🔍 DUPLICATE CHECKS
Duplicate rows (all columns): 0
Duplicate pose names: 0
Duplicate Sanskrit names: 30
Duplicate IDs: 0
✓ No duplicate pose names


In [72]:
# Check for missing or empty values
print(f"\n📋 MISSING/EMPTY VALUE CHECKS")
print("\nNull values per column:")
print(df_final.isnull().sum())

print("\nEmpty strings per column:")
for col in df_final.columns:
    if df_final[col].dtype == "object":
        empty_count = (df_final[col].str.strip() == "").sum()
        if empty_count > 0:
            print(f"  {col}: {empty_count}")
        else:
            print(f"  {col}: ✓ No empty strings")


📋 MISSING/EMPTY VALUE CHECKS

Null values per column:
id                   0
pose_name            0
sanskrit_name        0
category             0
difficulty_level     0
benefits             0
contraindications    0
breathing_pattern    0
duration_or_reps     0
modifications        0
instructions         0
dtype: int64

Empty strings per column:
  pose_name: ✓ No empty strings
  sanskrit_name: ✓ No empty strings
  category: ✓ No empty strings
  difficulty_level: ✓ No empty strings
  benefits: ✓ No empty strings
  contraindications: ✓ No empty strings
  breathing_pattern: ✓ No empty strings
  duration_or_reps: ✓ No empty strings
  modifications: ✓ No empty strings
  instructions: ✓ No empty strings


In [73]:
# Content length validation
print(f"\n📏 CONTENT LENGTH VALIDATION")

text_columns = [
    "benefits",
    "contraindications",
    "breathing_pattern",
    "duration_or_reps",
    "modifications",
    "instructions",
]

for col in text_columns:
    lengths = df_final[col].str.len()
    print(f"\n{col}:")
    print(f"  Min length: {lengths.min()} chars")
    print(f"  Max length: {lengths.max()} chars")
    print(f"  Avg length: {lengths.mean():.1f} chars")

    # Flag suspiciously short content
    short_threshold = (
        30 if col in ["duration_or_reps", "breathing_pattern"] else 50
    )
    short_count = (lengths < short_threshold).sum()
    if short_count > 0:
        print(
            f"  ⚠️ {short_count} poses with content < {short_threshold} chars"
        )
    else:
        print(f"  ✓ All content >= {short_threshold} chars")


📏 CONTENT LENGTH VALIDATION

benefits:
  Min length: 196 chars
  Max length: 423 chars
  Avg length: 295.7 chars
  ✓ All content >= 50 chars

contraindications:
  Min length: 100 chars
  Max length: 307 chars
  Avg length: 208.1 chars
  ✓ All content >= 50 chars

breathing_pattern:
  Min length: 49 chars
  Max length: 278 chars
  Avg length: 158.8 chars
  ✓ All content >= 30 chars

duration_or_reps:
  Min length: 40 chars
  Max length: 204 chars
  Avg length: 97.7 chars
  ✓ All content >= 30 chars

modifications:
  Min length: 128 chars
  Max length: 365 chars
  Avg length: 235.9 chars
  ✓ All content >= 50 chars

instructions:
  Min length: 293 chars
  Max length: 668 chars
  Avg length: 471.4 chars
  ✓ All content >= 50 chars


In [ ]:
# Category and difficulty validation
print(f"\n🏷️ CATEGORY & DIFFICULTY VALIDATION")

expected_categories = [
    "standing",
    "seated",
    "balancing",
    "inversion",
    "backbend",
    "forward_fold",
    "twist",
    "restorative",
    "lunge",
]
expected_difficulties = ["beginner", "intermediate", "advanced"]

# Check for invalid categories
invalid_categories = df_final[~df_final["category"].isin(expected_categories)]
if len(invalid_categories) > 0:
    print(f"⚠️ {len(invalid_categories)} poses with invalid categories")
else:
    print("✓ All categories valid")

# Check for invalid difficulties
invalid_difficulties = df_final[
    ~df_final["difficulty_level"].isin(expected_difficulties)
]
if len(invalid_difficulties) > 0:
    print(
        f"⚠️ {len(invalid_difficulties)} poses with invalid difficulty levels"
    )
else:
    print("✓ All difficulty levels valid")

print("\nCategory distribution:")
print(df_final["category"].value_counts().sort_index())

print("\nDifficulty distribution:")
print(df_final["difficulty_level"].value_counts())


🏷️ CATEGORY & DIFFICULTY VALIDATION
✓ All categories valid
✓ All difficulty levels valid

Category distribution:
category
backbend        20
balancing       33
forward_fold    26
inversion       20
lunge            3
restorative     20
seated          17
standing        38
twist           25
Name: count, dtype: int64

Difficulty distribution:
difficulty_level
intermediate    75
beginner        69
advanced        58
Name: count, dtype: int64


In [ ]:
# Final summary
print(f"\n{'='*70}")
print("FINAL SUMMARY")
print(f"{'='*70}")

issues = []

if df_final.duplicated().sum() > 0:
    issues.append(f"❌ {df_final.duplicated().sum()} duplicate rows")
if df_final["pose_name"].duplicated().sum() > 0:
    issues.append(
        f"❌ {df_final['pose_name'].duplicated().sum()} duplicate pose names"
    )
if df_final.isnull().sum().sum() > 0:
    issues.append(f"❌ {df_final.isnull().sum().sum()} null values")
if len(df_final) < 108:
    issues.append(f"❌ Only {len(df_final)} poses (target: 108+)")

if issues:
    print("\n⚠️ ISSUES FOUND:")
    for issue in issues:
        print(f"  {issue}")
    print("\n👉 Please review and fix the issues above.")
else:
    print("\n✅ ALL QUALITY CHECKS PASSED!")
    print(f"   - {len(df_final)} yoga poses")
    print(f"   - {len(df_final['category'].unique())} categories")
    print(
        f"   - {len(df_final['difficulty_level'].unique())} difficulty levels"
    )
    print(f"   - Location: ../data/yoga_data.csv")


FINAL SUMMARY

✅ ALL QUALITY CHECKS PASSED!
   - 202 yoga poses
   - 9 categories
   - 3 difficulty levels
   - Location: ../data/yoga_data.csv
